In [ ]:
import requests
from lxml import etree
import time
import csv

# 基础URL
base_url = "https://www.cuit.edu.cn"
news_list_url = base_url + "/index/cxyw.htm"
news_detail_url = base_url + "/info/1002/{}.htm"

# 添加请求头，模拟浏览器访问
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/93.0.4577.63 Safari/537.36'
}


# 获取网页内容
def get_page_content(url, retries=1):
    try:
        response = requests.get(url, headers=headers, timeout=10)
        response.encoding = 'utf-8'  # 确保编码正确
        if response.status_code == 200:
            return response.text
        else:
            print(f"Failed to fetch page: {url}, Status code: {response.status_code}")
            if retries > 0:
                print("Retrying...")
                return get_page_content(url, retries=retries - 1)
            return None
    except requests.RequestException as e:
        print(f"Request failed for {url}: {e}")
        if retries > 0:
            print("Retrying...")
            return get_page_content(url, retries=retries - 1)
        return None


# 解析要闻列表页，提取每篇文章的ID和标题链接
def parse_news_list(page_content):
    tree = etree.HTML(page_content)
    news_items = tree.xpath('/html/body/div[7]/div[5]/div/div[2]/div/ul/li/a')
    news_list = []

    for item in news_items:
        title = item.text.strip()
        href = item.get('href')
        # 处理相对路径，补充基础URL，并提取ID部分
        if not href.startswith("http"):
            href = base_url + href
        news_id = href.split('/')[-1].split('.')[0].split('?')[0]  # 提取ID部分，去掉可能的参数
        news_list.append((news_id, title))

    return news_list


# 解析要闻详情页，提取标题、来源、作者、日期和正文
def parse_news_detail(news_id):
    url = news_detail_url.format(news_id)
    page_content = get_page_content(url)

    if page_content:
        tree = etree.HTML(page_content)
        try:
            # 按照新的XPath结构提取数据
            title = tree.xpath('/html/body/div[7]/div[5]/div/form/h2/text()')
            title = title[0].strip() if title else 'NONE'  # 标题处理为空时设为NONE

            source = tree.xpath('/html/body/div[7]/div[5]/div/form/p/span[1]/text()')
            source = source[0].replace('来源：', '').strip() if source else 'NONE'  # 丢失来源时设为NONE

            author = tree.xpath('/html/body/div[7]/div[5]/div/form/p/span[2]/text()')
            author = author[0].replace('作者:', '').strip() if author else 'NONE'  # 丢失作者时设为NONE

            date = tree.xpath('/html/body/div[7]/div[5]/div/form/p/span[3]/text()')
            date = date[0].replace('日期：', '').strip() if date else 'NONE'  # 丢失日期时设为NONE

            # 提取正文，合并各段内容为完整文本
            paragraphs = tree.xpath('/html/body/div[7]/div[5]/div/form/div[1]//p/text()')
            content = '\n'.join([p.strip() for p in paragraphs if p.strip()])  # 合并段落，并去掉空白

            return {
                'id': news_id,
                'title': title,
                'source': source,
                'author': author,
                'date': date,
                'content': content
            }

        except Exception as e:
            print(f"Failed to parse news detail for ID: {news_id}, Error: {e}")
            return None
    return None


# 保存到CSV文件
def save_to_csv(news_data, filename='news_data.csv'):
    with open(filename, 'w', newline='', encoding='utf-8') as csvfile:
        fieldnames = ['id', 'title', 'source', 'author', 'date', 'content']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(news_data)


# 获取多个新闻列表页的URL（最多到第192页）
def get_news_list_urls(total_pages=192):
    urls = [news_list_url]  # 起始页
    for i in range(1, total_pages):  # 页码从1开始
        urls.append(f"{base_url}/index/cxyw/{i}.htm")
    return urls


# 主函数，控制爬取流程
def main():
    news_data = []
    total_news_to_scrape = 2000  # 爬取目标
    news_list_urls = get_news_list_urls(total_pages=192)  # 爬取192页

    for page_num, news_list_url in enumerate(news_list_urls, start=1):
        print(f"Fetching page {page_num}: {news_list_url}")
        page_content = get_page_content(news_list_url)

        if page_content:
            news_list = parse_news_list(page_content)

            for i, (news_id, title) in enumerate(news_list):
                print(f"Fetching news {i + 1} on page {page_num}: {title} (ID: {news_id})")

                # 尝试爬取新闻详情页，重试一次
                news_detail = parse_news_detail(news_id)

                if news_detail:
                    news_data.append(news_detail)

                # 控制请求频率，避免服务器负担过重
                time.sleep(1)  # 每次请求间隔1秒

                # 爬取2000条数据后停止
                if len(news_data) >= total_news_to_scrape:
                    save_to_csv(news_data)
                    print("Data saved to news_data.csv. Scraping completed.")
                    return  # 提前退出

    # 完成所有爬取后保存结果
    save_to_csv(news_data)
    print("Data saved to news_data.csv. Scraping completed.")


if __name__ == '__main__':
    main()


In [ ]:
# text_mining.py

import csv
import jieba
from collections import defaultdict
import os

def generate_text_mining_data(csv_file_path='news_data.csv', output_file_path='text_mining_data.txt'):
    """
    从CSV文件中提取id、source和content列，并保存到text_mining_data.txt文件中。
    格式为：id \t source \t content
    """
    if not os.path.exists(csv_file_path):
        print(f"错误：CSV文件 '{csv_file_path}' 不存在。")
        return
    
    with open(csv_file_path, 'r', encoding='utf-8') as csvfile, \
         open(output_file_path, 'w', encoding='utf-8') as txtfile:
        reader = csv.DictReader(csvfile)
        for row_num, row in enumerate(reader, start=1):
            id_ = row.get('id', '').strip()
            source = row.get('source', '').strip()
            content = row.get('content', '').strip().replace('\n', ' ')  # 替换换行符为空格
            if not id_ or not source or not content:
                print(f"警告：第 {row_num} 行数据不完整，已跳过。")
                continue
            txtfile.write(f"{id_}\t{source}\t{content}\n")
    print(f"已生成 '{output_file_path}' 文件。")

def index(src_file_path="text_mining_data.txt", index_file_path="indexed_text_mining_data.txt"):

    if not os.path.exists(src_file_path):
        print(f"错误：源文件 '{src_file_path}' 不存在。")
        return
    
    inverted_index = defaultdict(set)  # 使用set避免重复的文档ID

    with open(src_file_path, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f, start=1):
            parts = line.strip().split('\t')
            if len(parts) < 3:
                print(f"警告：第 {line_num} 行格式不正确，已跳过。")
                continue
            doc_id, source, content = parts
            # 使用jieba进行分词
            words = jieba.lcut(content)
            for word in words:
                word = word.strip()
                if word:  # 过滤空字符串
                    inverted_index[word].add(doc_id)
            if line_num % 100 == 0:
                print(f"已处理 {line_num} 行数据。")

    # 将倒排索引写入文件
    with open(index_file_path, 'w', encoding='utf-8') as f:
        for word, doc_ids in inverted_index.items():
            doc_id_list = ' '.join(doc_ids)
            f.write(f"{word}\t{doc_id_list}\n")
    print(f"倒排索引已保存至 '{index_file_path}'。")

def search(words, index_file_path="indexed_text_mining_data.txt"):

    if not os.path.exists(index_file_path):
        print(f"错误：索引文件 '{index_file_path}' 不存在。")
        return []
    
    # 读取倒排索引
    inverted_index = {}
    with open(index_file_path, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f, start=1):
            parts = line.strip().split('\t')
            if len(parts) != 2:
                print(f"警告：第 {line_num} 行索引格式不正确，已跳过。")
                continue
            word, doc_ids = parts
            inverted_index[word] = set(doc_ids.split())

    # 处理查询词
    if isinstance(words, str):
        query_words = jieba.lcut(words)
    elif isinstance(words, list):
        query_words = words
    else:
        print("错误：查询词格式不正确，应为字符串或字符串列表。")
        return []

    # 查找每个查询词对应的文档ID集合
    doc_id_sets = []
    for word in query_words:
        word = word.strip()
        if not word:
            continue
        if word in inverted_index:
            doc_id_sets.append(inverted_index[word])
        else:
            # 如果某个词不在索引中，说明没有匹配的文档
            print(f"提示：词语 '{word}' 不存在于索引中。")
            return []

    if not doc_id_sets:
        return []
    
    # 取交集，找到包含所有查询词的文档ID
    result_doc_ids = set.intersection(*doc_id_sets)
    return sorted(list(result_doc_ids))

def main():
    """
    主函数，控制数据处理和搜索流程。
    """
    print("文本挖掘系统启动。")
    print("请选择操作：")
    print("1. 生成 text_mining_data.txt")
    print("2. 构建倒排索引")
    print("3. 搜索文档")
    print("4. 全部执行（生成数据、构建索引、进入搜索）")
    print("5. 退出")

    while True:
        choice = input("\n请输入操作编号（1-5）：").strip()
        
        if choice == '1':
            generate_text_mining_data()
        elif choice == '2':
            index()
        elif choice == '3':
            if not os.path.exists("indexed_text_mining_data.txt"):
                print("错误：倒排索引文件不存在，请先执行构建倒排索引操作。")
                continue
            query = input("请输入要搜索的词语（支持多个词，用空格分隔）：").strip()
            if not query:
                print("提示：请输入有效的查询词。")
                continue
            result = search(query)
            if result:
                print(f"包含所有查询词的文档ID列表（共 {len(result)} 个）：")
                print(', '.join(result))
            else:
                print("没有找到匹配的文档。")
        elif choice == '4':
            generate_text_mining_data()
            index()
            if not os.path.exists("indexed_text_mining_data.txt"):
                print("错误：倒排索引构建失败。")
                continue
            while True:
                query = input("\n请输入要搜索的词语（支持多个词，用空格分隔，输入 '退出' 结束）：").strip()
                if query.lower() == '退出':
                    break
                if not query:
                    print("提示：请输入有效的查询词。")
                    continue
                result = search(query)
                if result:
                    print(f"包含所有查询词的文档ID列表（共 {len(result)} 个）：")
                    print(', '.join(result))
                else:
                    print("没有找到匹配的文档。")
        elif choice == '5':
            print("退出程序。")
            break
        else:
            print("无效的输入，请输入1-5之间的数字。")

if __name__ == '__main__':
    main()


In [ ]:
import os
import jieba
import csv
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def vectorize(src_file_path="text_mining_data.txt", dest_file_path="vectorized_text_mining_data.txt"):

    if not os.path.exists(src_file_path):
        print(f"错误：源文件 '{src_file_path}' 不存在。")
        return
    
    doc_ids = []
    documents = []
    
    with open(src_file_path, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f, start=1):
            parts = line.strip().split('\t')
            if len(parts) < 3:
                print(f"警告：第 {line_num} 行格式不正确，已跳过。")
                continue
            doc_id, source, content = parts
            # 使用jieba进行分词
            tokens = jieba.lcut(content)
            # 连接分词结果
            tokenized_content = ' '.join(tokens)
            doc_ids.append(doc_id)
            documents.append(tokenized_content)
            if line_num % 500 == 0:
                print(f"已处理 {line_num} 行数据。")
    
    # 使用TfidfVectorizer进行向量化
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(documents)
    # 保存向量化结果
    # 每行对应一个文档，格式为：docID \t word1:tfidf1 word2:tfidf2 ...
    feature_names = vectorizer.get_feature_names_out()
    
    with open(dest_file_path, 'w', encoding='utf-8') as f:
        for doc_index, doc_id in enumerate(doc_ids):
            # 获取非零TF-IDF值的索引
            non_zero_indices = tfidf_matrix[doc_index].nonzero()[1]
            # 构建word:tfidf字符串
            word_tfidf_pairs = []
            for idx in non_zero_indices:
                word = feature_names[idx]
                tfidf_value = tfidf_matrix[doc_index, idx]
                word_tfidf_pairs.append(f"{word}:{tfidf_value:.4f}")
            # 写入文件
            f.write(f"{doc_id}\t{' '.join(word_tfidf_pairs)}\n")
            if (doc_index + 1) % 500 == 0:
                print(f"已向量化 {doc_index + 1} 个文档。")
    
    print(f"向量化结果已保存至 '{dest_file_path}'。")

def cosine_similarity_calculation(vector_file_path="vectorized_text_mining_data.txt", 
                                 cos_sim_file_path="cos_sim_matrix.csv"):

    if not os.path.exists(vector_file_path):
        print(f"错误：向量化文件 '{vector_file_path}' 不存在。")
        return
    
    doc_ids = []
    vectors = []
    
    with open(vector_file_path, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f, start=1):
            parts = line.strip().split('\t')
            if len(parts) < 2:
                print(f"警告：第 {line_num} 行格式不正确，已跳过。")
                continue
            doc_id, word_tfidf = parts
            doc_ids.append(doc_id)
            # 解析word:tfidf
            tfidf_values = {}
            for pair in word_tfidf.split():
                word, tfidf = pair.split(':')
                tfidf_values[word] = float(tfidf)
            vectors.append(tfidf_values)
            if line_num % 500 == 0:
                print(f"已读取向量化数据 {line_num} 行。")
    
    # 构建词汇表
    vocabulary = set()
    for vec in vectors:
        vocabulary.update(vec.keys())
    vocabulary = sorted(vocabulary)
    word_index = {word: idx for idx, word in enumerate(vocabulary)}
    print(f"词汇表大小：{len(vocabulary)}")
    
    # 构建TF-IDF矩阵
    import numpy as np
    tfidf_matrix = np.zeros((len(vectors), len(vocabulary)))
    
    for doc_idx, vec in enumerate(vectors):
        for word, tfidf in vec.items():
            if word in word_index:
                tfidf_matrix[doc_idx, word_index[word]] = tfidf
        if (doc_idx + 1) % 500 == 0:
            print(f"已构建 TF-IDF 矩阵 {doc_idx + 1} 行。")
    
    # 计算余弦相似度
    print("开始计算余弦相似度矩阵...")
    cos_sim = cosine_similarity(tfidf_matrix)
    print("余弦相似度计算完成。")
    
    with open(cos_sim_file_path, 'w', encoding='utf-8', newline='') as csvfile:
        writer = csv.writer(csvfile)
        # 写入表头
        writer.writerow(['docID'] + doc_ids)
        # 写入每行
        for i, doc_id in enumerate(doc_ids):
            writer.writerow([doc_id] + list(cos_sim[i]))
            if (i + 1) % 500 == 0:
                print(f"已写入余弦相似度数据 {i + 1} 行。")
    
    print(f"余弦相似度矩阵已保存至 '{cos_sim_file_path}'。")

def get_top_n():
    import pandas as pd
    topn = 10
    # 读取余弦相似度矩阵
    cos_sim_df = pd.read_csv('cos_sim_matrix.csv', index_col=0)
    cos_sim_df.index = cos_sim_df.index.astype(str)
    cos_sim_df.columns = cos_sim_df.columns.astype(str)
    # 读取文档来源
    doc_source = {}
    with open('news_data.csv', 'r', encoding='utf-8-sig') as f_csv:
        reader = csv.DictReader(f_csv)
        for i, row in enumerate(reader, 1):
            doc_id = str(row.get('id', '').strip())
            source = row.get('source', '').strip()
            if doc_id and source:
                doc_source[doc_id] = source
            if i % 500 == 0:
                print(f"已读取文档来源 {i} 行。")
    # 获取每个文档的 Top N 相似文档
    with open('top_n_similar_docs.txt', 'w', encoding='utf-8') as f_out:
        for doc_id in cos_sim_df.index:
            doc_id = str(doc_id)
            sim_scores = cos_sim_df.loc[doc_id]
            sim_scores.index = sim_scores.index.astype(str)
            # 排序，降序，排除自身
            sim_scores = sim_scores.drop(labels=[doc_id], errors='ignore')
            top_n_docs = sim_scores.nlargest(topn)
            # 获取 Top N 文档的 ID 和来源
            top_n_info = []
            for other_doc_id, score in top_n_docs.items():
                source = doc_source.get(other_doc_id, '未知来源')
                top_n_info.append(f"{other_doc_id}:{source}:{score:.4f}")
            source = doc_source.get(doc_id, '未知来源')
            f_out.write(f"{doc_id}:{source}\t{' '.join(top_n_info)}\n")
    print(f"Top {topn} 相似文档已保存至 'top_n_similar_docs.txt'。")


def main():
    """
    主函数，控制任务2的执行流程。
    """
    print("任务2：向量空间模型和词袋模型应用")
    print("请选择子任务：")
    print("1. 向量化正文内容")
    print("2. 计算余弦相似度矩阵")
    print("3. 获取Top N相似文档")
    print("4. 全部执行（向量化、计算相似度、获取Top N）")
    print("5. 退出")
    
    while True:
        choice = input("\n请输入操作编号（1-5）：").strip()
        
        if choice == '1':
            vectorize()
        elif choice == '2':
            cosine_similarity_calculation()
        elif choice == '3':
            get_top_n()
        elif choice == '4':
            vectorize()
            cosine_similarity_calculation()
            get_top_n()
        elif choice == '5':
            print("退出程序。")
            break
        else:
            print("无效的输入，请输入1-5之间的数字。")

if __name__ == '__main__':
    main()


In [ ]:
# text_classification.py

import os
import random
import csv
import jieba
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score, precision_recall_fscore_support
import numpy as np

def divide(src_file_path, training_data_path, test_data_path, test_size=0.2, random_seed=42):
    """
    实现数据划分功能，将数据集划分为训练集和测试集。

    参数：
    - src_file_path: str，源文件路径，默认为"text_mining_data.txt"
    - training_data_path: str，训练集保存路径
    - test_data_path: str，测试集保存路径
    - test_size: float，测试集占比，默认为0.2（20%）
    - random_seed: int，随机种子，确保结果可复现
    """
    if not os.path.exists(src_file_path):
        print(f"错误：源文件 '{src_file_path}' 不存在。")
        return

    data = []
    with open(src_file_path, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f, start=1):
            parts = line.strip().split('\t')
            if len(parts) < 3:
                print(f"警告：第 {line_num} 行格式不正确，已跳过。")
                continue
            doc_id, source, content = parts
            data.append((doc_id, source, content))

    # 打乱数据
    random.seed(random_seed)
    random.shuffle(data)

    # 划分训练集和测试集
    split_index = int(len(data) * (1 - test_size))
    training_data = data[:split_index]
    test_data = data[split_index:]

    # 保存训练集
    with open(training_data_path, 'w', encoding='utf-8', newline='') as f:
        writer = csv.writer(f, delimiter='\t')
        writer.writerow(['id', 'source', 'content'])
        writer.writerows(training_data)
    print(f"训练集已保存至 '{training_data_path}'。共 {len(training_data)} 条记录。")

    # 保存测试集
    with open(test_data_path, 'w', encoding='utf-8', newline='') as f:
        writer = csv.writer(f, delimiter='\t')
        writer.writerow(['id', 'source', 'content'])
        writer.writerows(test_data)
    print(f"测试集已保存至 '{test_data_path}'。共 {len(test_data)} 条记录。")

def vectorize(training_data_path, vectorized_training_data_path, dict_path, vectorizer_path):
    """
    实现向量化功能，将训练集的文本内容转换为TF-IDF向量。

    参数：
    - training_data_path: str，训练集文件路径
    - vectorized_training_data_path: str，向量化训练集保存路径
    - dict_path: str，词典保存路径
    - vectorizer_path: str，向量化模型保存路径
    """
    if not os.path.exists(training_data_path):
        print(f"错误：训练集文件 '{training_data_path}' 不存在。")
        return

    doc_ids = []
    labels = []
    documents = []

    with open(training_data_path, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f, delimiter='\t')
        for row_num, row in enumerate(reader, start=1):
            doc_id = row['id'].strip()
            source = row['source'].strip()
            content = row['content'].strip().replace('\n', ' ')
            tokens = jieba.lcut(content)
            tokenized_content = ' '.join(tokens)
            doc_ids.append(doc_id)
            labels.append(source)
            documents.append(tokenized_content)
            if row_num % 500 == 0:
                print(f"已处理 {row_num} 行训练数据。")

    # 使用TfidfVectorizer进行向量化
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform(documents)

    # 保存词典
    with open(dict_path, 'w', encoding='utf-8') as f:
        for word in vectorizer.get_feature_names_out():
            f.write(f"{word}\n")
    print(f"词典已保存至 '{dict_path}'。共 {len(vectorizer.get_feature_names_out())} 个词。")

    # 保存向量化训练集
    np.save(vectorized_training_data_path, tfidf_matrix.toarray())
    print(f"向量化训练集已保存至 '{vectorized_training_data_path}'。")

    # 保存向量化模型
    import joblib
    joblib.dump(vectorizer, vectorizer_path)
    print(f"向量化模型已保存至 '{vectorizer_path}'。")

def train(vectorized_training_data_path, labels, model_path):
    """
    实现训练功能，训练最大熵模型（逻辑回归）。

    参数：
    - vectorized_training_data_path: str，向量化训练集文件路径
    - labels: list，训练集标签
    - model_path: str，训练好的模型保存路径
    """
    if not os.path.exists(vectorized_training_data_path):
        print(f"错误：向量化训练集文件 '{vectorized_training_data_path}' 不存在。")
        return

    # 加载向量化训练集
    tfidf_matrix = np.load(vectorized_training_data_path)

    # 训练最大熵模型（逻辑回归）
    model = LogisticRegression(max_iter=1000, multi_class='multinomial', solver='lbfgs')
    model.fit(tfidf_matrix, labels)
    print("最大熵模型训练完成。")

    # 保存模型
    import joblib
    joblib.dump(model, model_path)
    print(f"训练好的模型已保存至 '{model_path}'。")

def test(test_data_path, dict_path, vectorizer_path, model_path, answer_path):
    """
    实现测试功能，使用训练好的模型对测试集进行预测。

    参数：
    - test_data_path: str，测试集文件路径
    - dict_path: str，词典文件路径
    - vectorizer_path: str，向量化模型文件路径
    - model_path: str，训练好的模型文件路径
    - answer_path: str，预测结果保存路径
    """
    if not os.path.exists(test_data_path):
        print(f"错误：测试集文件 '{test_data_path}' 不存在。")
        return
    if not os.path.exists(vectorizer_path):
        print(f"错误：向量化模型文件 '{vectorizer_path}' 不存在。")
        return
    if not os.path.exists(model_path):
        print(f"错误：训练好的模型文件 '{model_path}' 不存在。")
        return

    # 加载向量化模型
    import joblib
    vectorizer = joblib.load(vectorizer_path)

    # 加载训练好的模型
    model = joblib.load(model_path)

    doc_ids = []
    true_labels = []
    documents = []

    with open(test_data_path, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f, delimiter='\t')
        for row_num, row in enumerate(reader, start=1):
            doc_id = row['id'].strip()
            source = row['source'].strip()
            content = row['content'].strip().replace('\n', ' ')
            tokens = jieba.lcut(content)
            tokenized_content = ' '.join(tokens)
            doc_ids.append(doc_id)
            true_labels.append(source)
            documents.append(tokenized_content)
            if row_num % 500 == 0:
                print(f"已处理 {row_num} 行测试数据。")

    # 向量化测试集
    tfidf_matrix = vectorizer.transform(documents)

    # 进行预测
    predictions = model.predict(tfidf_matrix)
    print("测试集预测完成。")

    # 保存预测结果
    with open(answer_path, 'w', encoding='utf-8') as f:
        writer = csv.writer(f, delimiter='\t')
        writer.writerow(['id', 'true_label', 'predicted_label'])
        for doc_id, true_label, pred_label in zip(doc_ids, true_labels, predictions):
            writer.writerow([doc_id, true_label, pred_label])
    print(f"预测结果已保存至 '{answer_path}'。")

def evaluate(answer_path, result_path):
    """
    实现评估功能，计算混淆矩阵、准确率、准确率（Precision）、召回率（Recall）和F度量值（F-Measure）。

    参数：
    - answer_path: str，预测结果文件路径
    - result_path: str，评估结果保存路径
    """
    if not os.path.exists(answer_path):
        print(f"错误：预测结果文件 '{answer_path}' 不存在。")
        return

    doc_ids = []
    true_labels = []
    predicted_labels = []

    with open(answer_path, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f, delimiter='\t')
        for row in reader:
            doc_ids.append(row['id'])
            true_labels.append(row['true_label'])
            predicted_labels.append(row['predicted_label'])

    # 获取所有类别
    classes = sorted(list(set(true_labels + predicted_labels)))

    # 计算混淆矩阵
    cm = confusion_matrix(true_labels, predicted_labels, labels=classes)

    # 计算准确率（Accuracy）
    accuracy = accuracy_score(true_labels, predicted_labels)

    # 计算每个类别的Precision, Recall, F-Measure
    precision, recall, f_measure, _ = precision_recall_fscore_support(true_labels, predicted_labels, labels=classes, zero_division=0)

    # 写入评估结果
    with open(result_path, 'w', encoding='utf-8') as f:
        f.write(f"Accuracy={accuracy:.4f}\n")
        for idx, cls in enumerate(classes, start=1):
            f.write(f"类别{idx}\tPrecision={precision[idx-1]:.4f}\tRecall={recall[idx-1]:.4f}\tF-Measure={f_measure[idx-1]:.4f}\n")
    print(f"评估结果已保存至 '{result_path}'。")

def main():
    """
    主函数，控制任务3的执行流程。
    """
    print("任务3：基于有监督学习的文本分类")
    print("请选择操作：")
    print("1. 划分数据集")
    print("2. 向量化训练集")
    print("3. 训练模型")
    print("4. 测试模型")
    print("5. 评估模型")
    print("6. 全部执行（划分数据、向量化、训练、测试、评估）")
    print("7. 退出")

    # 定义文件路径
    src_file_path = "text_mining_data.txt"
    training_data_path = "training_data.txt"
    test_data_path = "test_data.txt"
    vectorized_training_data_path = "vectorized_training_data.npy"
    dict_path = "vocab.txt"
    vectorizer_path = "tfidf_vectorizer.pkl"
    model_path = "maxent_model.pkl"
    answer_path = "predictions.txt"
    result_path = "eval_result.txt"

    while True:
        choice = input("\n请输入操作编号（1-7）：").strip()

        if choice == '1':
            divide(src_file_path, training_data_path, test_data_path)
        elif choice == '2':
            # 读取训练集标签
            labels = []
            with open(training_data_path, 'r', encoding='utf-8') as f:
                reader = csv.DictReader(f, delimiter='\t')
                for row in reader:
                    labels.append(row['source'].strip())
            vectorize(training_data_path, vectorized_training_data_path, dict_path, vectorizer_path)
        elif choice == '3':
            # 读取训练集标签
            labels = []
            with open(training_data_path, 'r', encoding='utf-8') as f:
                reader = csv.DictReader(f, delimiter='\t')
                for row in reader:
                    labels.append(row['source'].strip())
            train(vectorized_training_data_path, labels, model_path)
        elif choice == '4':
            test(test_data_path, dict_path, vectorizer_path, model_path, answer_path)
        elif choice == '5':
            evaluate(answer_path, result_path)
        elif choice == '6':
            # 划分数据集
            divide(src_file_path, training_data_path, test_data_path)
            # 读取训练集标签
            labels = []
            with open(training_data_path, 'r', encoding='utf-8') as f:
                reader = csv.DictReader(f, delimiter='\t')
                for row in reader:
                    labels.append(row['source'].strip())
            # 向量化训练集
            vectorize(training_data_path, vectorized_training_data_path, dict_path, vectorizer_path)
            # 训练模型
            train(vectorized_training_data_path, labels, model_path)
            # 测试模型
            test(test_data_path, dict_path, vectorizer_path, model_path, answer_path)
            # 评估模型
            evaluate(answer_path, result_path)
        elif choice == '7':
            print("退出程序。")
            break
        else:
            print("无效的输入，请输入1-7之间的数字。")

if __name__ == '__main__':
    main()


In [ ]:
import os
import csv
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, accuracy_score, precision_recall_fscore_support

# 设置中文字体以正常显示中文标签
plt.rcParams['font.sans-serif'] = ['SimHei']  
plt.rcParams['axes.unicode_minus'] = False  

def load_predictions(predictions_path):
    """
    读取预测结果文件，返回真实标签和预测标签列表。
    """
    true_labels = []
    predicted_labels = []
    
    with open(predictions_path, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f, delimiter='\t')
        for row in reader:
            true_labels.append(row['true_label'])
            predicted_labels.append(row['predicted_label'])
    
    return true_labels, predicted_labels

def plot_confusion_matrix(true_labels, predicted_labels, classes, result_image_path='confusion_matrix.png'):
    """
    绘制并保存混淆矩阵，同时将矩阵数据保存为文本文件。
    """
    cm = confusion_matrix(true_labels, predicted_labels, labels=classes)
    cm_df = pd.DataFrame(cm, index=classes, columns=classes)
    
    # 绘制混淆矩阵
    plt.figure(figsize=(10, 7))
    sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues')
    plt.title('混淆矩阵')
    plt.ylabel('真实类别')
    plt.xlabel('预测类别')
    plt.savefig(result_image_path)
    plt.close()
    print(f"混淆矩阵已保存至 '{result_image_path}'。")

    # 将混淆矩阵数据保存为文本文件
    cm_txt_path = 'confusion_matrix.txt'
    cm_df.to_csv(cm_txt_path, sep='\t', index=True)
    print(f"混淆矩阵数据已保存至 '{cm_txt_path}'。")
    
def calculate_metrics(true_labels, predicted_labels, classes):
    """
    计算准确率、精确率、召回率和F1分数。
    
    返回：
    - accuracy: float，整体准确率
    - per_class_metrics: dict，每个类别的Precision、Recall和F1-Measure
    """
    accuracy = accuracy_score(true_labels, predicted_labels)
    precision, recall, f1, _ = precision_recall_fscore_support(true_labels, predicted_labels, labels=classes, zero_division=0)
    
    per_class_metrics = {}
    for idx, cls in enumerate(classes):
        per_class_metrics[cls] = {
            'Precision': precision[idx],
            'Recall': recall[idx],
            'F1-Measure': f1[idx]
        }
    
    return accuracy, per_class_metrics

def save_evaluation_results(accuracy, per_class_metrics, classes, eval_result_path='eval_result.txt'):
    """
    将评估结果保存到文件中。
    
    参数：
    - accuracy: float，整体准确率
    - per_class_metrics: dict，每个类别的Precision、Recall和F1-Measure
    - classes: list，类别名称
    - eval_result_path: str，评估结果保存路径
    """
    with open(eval_result_path, 'w', encoding='utf-8') as f:
        f.write(f"Accuracy={accuracy:.4f}\n")
        for idx, cls in enumerate(classes, start=1):
            metrics = per_class_metrics[cls]
            f.write(f"{cls}\tPrecision={metrics['Precision']:.4f}\tRecall={metrics['Recall']:.4f}\tF-Measure={metrics['F1-Measure']:.4f}\n")
    print(f"评估结果已保存至 '{eval_result_path}'。")

def main():
    # 定义文件路径
    predictions_path = 'predictions.txt'
    confusion_matrix_image = 'confusion_matrix.png'
    eval_result_path = 'eval_result.txt'
    
    # 检查文件是否存在
    if not os.path.exists(predictions_path):
        print(f"错误：预测结果文件 '{predictions_path}' 不存在。")
        return
    
    # 加载预测结果
    true_labels, predicted_labels = load_predictions(predictions_path)
    
    # 获取所有类别（排序以确保一致性）
    classes = sorted(list(set(true_labels + predicted_labels)))
    
    # 绘制并保存混淆矩阵，同时保存为文本文件
    plot_confusion_matrix(true_labels, predicted_labels, classes, confusion_matrix_image)
    
    # 计算评估指标
    accuracy, per_class_metrics = calculate_metrics(true_labels, predicted_labels, classes)
    
    # 保存评估结果
    save_evaluation_results(accuracy, per_class_metrics, classes, eval_result_path)

if __name__ == '__main__':
    main()


In [3]:
import random

def shuffler(instances):
    n = len(instances)
    for i in range(n-1, 0, -1):
        j = random.randint(0, i)
        instances[i], instances[j] = instances[j], instances[i]
    return instances

def partition(instances, shuffler_func, proportion):
    if not isinstance(instances, list):
        raise TypeError("instances应为列表类型。")
    if not callable(shuffler_func):
        raise TypeError("shuffler应为可调用的函数。")
    if not isinstance(proportion, list) or not all(isinstance(p, (int, float)) for p in proportion):
        raise TypeError("proportion应为数字列表。")

    total = sum(proportion)
    if not total > 0:
        raise ValueError("proportion的总和应大于0。")
    
    # 归一化比例
    normalized_proportion = [p / total for p in proportion]
    
    # 计算每个部分的数量
    total_instances = len(instances)
    counts = [int(p * total_instances) for p in normalized_proportion]
    
    # 处理舍入误差，确保总数一致
    while sum(counts) < total_instances:
        # 找到最大比例对应的索引，增加1
        max_index = normalized_proportion.index(max(normalized_proportion))
        counts[max_index] += 1
    while sum(counts) > total_instances:
        # 找到最小比例对应的索引，减少1
        min_index = normalized_proportion.index(min(normalized_proportion))
        counts[min_index] -= 1
    
    # 洗牌
    shuffled_instances = shuffler_func(instances.copy())
    
    # 划分
    partitioned_instances = []
    start = 0
    for count in counts:
        end = start + count
        partitioned_instances.append(shuffled_instances[start:end])
        start = end
    
    return partitioned_instances

def main():
    # 示例输入
    instances = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
    proportion = [0.7, 0.1, 0.2]  # 划分比例

    print(f"原始数组: {instances}")
    
    # 洗牌
    shuffled = shuffler(instances.copy())
    print(f"洗牌后的数组: {shuffled}")
    
    # 划分
    partitioned = partition(instances, shuffler, proportion)
    print(f"按比例 {proportion} 划分后的数组: {partitioned}")
    
    # 验证示例输出
    # 由于洗牌是随机的，示例输出可能不同
    # 例如，输出可能为：
    # [[1, 3, 4, 6, 7, 9, 10], [8], [2, 5]]
    
if __name__ == "__main__":
    main()


原始数组: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
洗牌后的数组: [9, 5, 7, 10, 3, 8, 2, 6, 4, 1]
按比例 [0.7, 0.1, 0.2] 划分后的数组: [[3, 4, 2, 8, 7, 10, 5], [1], [6, 9]]


In [1]:
import os
import jieba
import numpy as np
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, adjusted_rand_score, confusion_matrix, accuracy_score, precision_recall_fscore_support
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns

def vectorize(src_file_path="text_mining_data.txt", vectorized_data_path="vectorized_data.npz", dict_path="dict.txt"):
    if not os.path.exists(src_file_path):
        print(f"错误：源文件 '{src_file_path}' 不存在。")
        return

    documents = []
    doc_ids = []
    true_labels = []
    with open(src_file_path, 'r', encoding='utf-8') as f:
        next(f)  # 跳过表头
        for line_num, line in enumerate(f, start=2):
            line = line.strip()
            if not line:
                print(f"警告：第 {line_num} 行为空，已跳过。")
                continue
            parts = line.split('\t')
            if len(parts) < 3:
                print(f"警告：第 {line_num} 行格式不正确，已跳过。")
                continue
            doc_id, source, content = parts
            # 分词
            tokens = jieba.lcut(content)
            # 连接分词结果
            tokenized_content = ' '.join(tokens)
            doc_ids.append(doc_id)
            true_labels.append(source)
            documents.append(tokenized_content)
            if (line_num - 1) % 500 == 0:
                print(f"已处理 {line_num - 1} 行数据。")

    vectorizer = TfidfVectorizer()
    X_sparse = vectorizer.fit_transform(documents)
    X = X_sparse.toarray()  # 转换为密集数组

    vocab = vectorizer.get_feature_names_out()
    with open(dict_path, 'w', encoding='utf-8') as f_dict:
        for word in vocab:
            f_dict.write(f"{word}\n")

    np.savez(vectorized_data_path, X=X, doc_ids=doc_ids, true_labels=true_labels)
    print(f"向量化完成，结果已保存到 '{vectorized_data_path}'，词典已保存到 '{dict_path}'。")

def compare_clustering_algorithms(vectorized_data_path="vectorized_data.npz"):
    if not os.path.exists(vectorized_data_path):
        print(f"错误：向量化数据文件 '{vectorized_data_path}' 不存在。")
        return

    data = np.load(vectorized_data_path, allow_pickle=True)
    X = data['X']
    true_labels = data['true_labels']

    algorithms = {
        "KMeans": KMeans(n_clusters=4, random_state=42),
        "Agglomerative": AgglomerativeClustering(n_clusters=4),
        "DBSCAN": DBSCAN(eps=0.5, min_samples=5)
    }

    best_algorithm = None
    best_score = -1
    best_labels = None

    for name, algorithm in algorithms.items():
        print(f"\n正在应用算法: {name}")
        if name == "DBSCAN":
            labels = algorithm.fit_predict(X)
        else:
            labels = algorithm.fit_predict(X)
        
        # 计算轮廓系数
        if len(set(labels)) > 1 and len(set(labels)) < len(X):
            score = silhouette_score(X, labels)
        else:
            score = -1  # 无法计算轮廓系数

        print(f"{name} Silhouette Score: {score}")

        if score > best_score:
            best_score = score
            best_algorithm = name
            best_labels = labels

    print(f"\n最佳算法是: {best_algorithm}，轮廓系数得分: {best_score}")

    return best_algorithm, best_labels, data['doc_ids'], true_labels

def map_clusters_to_labels(best_algorithm, best_labels, doc_ids, true_labels):
    if best_algorithm not in ["KMeans", "Agglomerative"]:
        print(f"当前算法 {best_algorithm} 不支持映射到特定类别。")
        return

    n_clusters = 4
    label_mapping = {}
    for cluster in range(n_clusters):
        # 获取属于该簇的所有样本的真实标签
        cluster_true_labels = [true_labels[i] for i in range(len(best_labels)) if best_labels[i] == cluster]
        if cluster_true_labels:
            # 统计每个标签的出现频率
            most_common_label = max(set(cluster_true_labels), key=cluster_true_labels.count)
            label_mapping[cluster] = most_common_label
        else:
            label_mapping[cluster] = "未知"

    # 显示簇与类别的映射关系
    print("\n簇与实际类别的映射关系：")
    for cluster, label in label_mapping.items():
        print(f"簇 {cluster}: {label}")

    # 将聚类标签映射为实际类别
    predicted_categories = [label_mapping.get(cluster, "未知") for cluster in best_labels]

    # 显示映射后的预测类别
    print("\n聚类结果对应的预测类别：")
    for i, (doc_id, pred) in enumerate(zip(doc_ids, predicted_categories)):
        print(f"文档ID: {doc_id} --> 预测类别: {pred}")

    return predicted_categories

def save_cluster_result(doc_ids, predicted_categories, answer_path="cluster_result.txt"):
    with open(answer_path, 'w', encoding='utf-8') as f_out:
        f_out.write("doc_id\tcluster_label\n")
        for doc_id, cluster_label in zip(doc_ids, predicted_categories):
            f_out.write(f"{doc_id}\t{cluster_label}\n")
    print(f"聚类结果已保存到 '{answer_path}'。")

def evaluate(answer_path="cluster_result.txt", src_file_path="text_mining_data.txt", result_path="eval_result.txt"):
    if not os.path.exists(answer_path):
        print(f"错误：聚类结果文件 '{answer_path}' 不存在。")
        return
    if not os.path.exists(src_file_path):
        print(f"错误：源文件 '{src_file_path}' 不存在。")
        return

    true_labels = []
    pred_labels = []

    # 创建 doc_id 到 source 的映射
    doc_id_to_source = {}
    with open(src_file_path, 'r', encoding='utf-8') as f_src:
        next(f_src)  # 跳过表头
        for line_num, line in enumerate(f_src, start=2):
            line = line.strip()
            if not line:
                continue
            parts = line.split('\t')
            if len(parts) < 3:
                continue
            doc_id, source, content = parts
            doc_id_to_source[doc_id] = source

    with open(answer_path, 'r', encoding='utf-8') as f:
        next(f)  # 跳过表头
        for line in f:
            doc_id, cluster_label = line.strip().split('\t')
            pred_labels.append(int(cluster_label))
            # 获取真实标签
            source = doc_id_to_source.get(doc_id, "未知")
            true_labels.append(source)

    # 编码标签
    le_true = LabelEncoder()
    true_labels_encoded = le_true.fit_transform(true_labels)

    # 聚类标签已是整数，无需编码
    predicted_labels_encoded = np.array(pred_labels)

    # 计算调整后的兰德指数
    ari_score = adjusted_rand_score(true_labels_encoded, predicted_labels_encoded)
    print(f"调整后的兰德指数 (Adjusted Rand Index): {ari_score}")

    # 写入评估结果
    with open(result_path, 'w', encoding='utf-8') as f_out:
        f_out.write(f"Adjusted Rand Index (ARI) = {ari_score:.4f}\n")
    print(f"评估结果已保存至 '{result_path}'。")

    # 可视化混淆矩阵
    cm = confusion_matrix(true_labels_encoded, predicted_labels_encoded)
    classes = le_true.classes_

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=[f'聚簇{i+1}' for i in range(cm.shape[1])],
                yticklabels=classes)
    plt.xlabel('预测聚簇')
    plt.ylabel('真实类别')
    plt.title('混淆矩阵')
    plt.savefig('confusion_matrix.png')
    plt.show()
    print("混淆矩阵已保存为 'confusion_matrix.png'。")

def main():
    try:
        # 第一步：向量化
        vectorize(src_file_path="text_mining_data.txt", vectorized_data_path="vectorized_data.npz", dict_path="dict.txt")
        
        # 第二步：比较聚类算法并选择最佳
        best_algorithm, best_labels, doc_ids, true_labels = compare_clustering_algorithms(vectorized_data_path="vectorized_data.npz")
        
        # 第三步：映射聚类标签到实际类别
        predicted_categories = map_clusters_to_labels(best_algorithm, best_labels, doc_ids, true_labels)
        
        # 第四步：保存聚类结果
        save_cluster_result(doc_ids, predicted_categories, answer_path="cluster_result.txt")
        
        # 第五步：评估聚类效果
        evaluate(answer_path="cluster_result.txt", src_file_path="text_mining_data.txt", result_path="eval_result.txt")
        
    except Exception as e:
        print(f"发生异常：{e}")

if __name__ == '__main__':
    main()


Building prefix dict from the default dictionary ...
Loading model from cache C:\Users\admin\AppData\Local\Temp\jieba.cache
Loading model cost 0.500 seconds.
Prefix dict has been built successfully.


已处理 500 行数据。
已处理 1000 行数据。
已处理 1500 行数据。
向量化完成，结果已保存到 'vectorized_data.npz'，词典已保存到 'dict.txt'。

正在应用算法: KMeans


c:\Users\admin\Desktop\py代码\venv\Lib\site-packages\sklearn\cluster\_kmeans.py:1412: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  super()._check_params_vs_input(X, default_n_init=10)


KMeans Silhouette Score: 0.009592825683089388

正在应用算法: Agglomerative
Agglomerative Silhouette Score: 0.0013372818323290376

正在应用算法: DBSCAN
DBSCAN Silhouette Score: -1

最佳算法是: KMeans，轮廓系数得分: 0.009592825683089388

簇与实际类别的映射关系：
簇 0: 学生工作处
簇 1: 科技处
簇 2: 教务处
簇 3: 国际交流与合作处

聚类结果对应的预测类别：
文档ID: 13158 --> 预测类别: 国际交流与合作处
文档ID: 13144 --> 预测类别: 国际交流与合作处
文档ID: 13145 --> 预测类别: 国际交流与合作处
文档ID: 13134 --> 预测类别: 学生工作处
文档ID: 13140 --> 预测类别: 国际交流与合作处
文档ID: 13121 --> 预测类别: 国际交流与合作处
文档ID: 13124 --> 预测类别: 教务处
文档ID: 13119 --> 预测类别: 国际交流与合作处
文档ID: 13097 --> 预测类别: 学生工作处
文档ID: 13093 --> 预测类别: 学生工作处
文档ID: 13088 --> 预测类别: 教务处
文档ID: 13089 --> 预测类别: 教务处
文档ID: 13087 --> 预测类别: 科技处
文档ID: 12314 --> 预测类别: 教务处
文档ID: 12315 --> 预测类别: 国际交流与合作处
文档ID: 12316 --> 预测类别: 教务处
文档ID: 12317 --> 预测类别: 国际交流与合作处
文档ID: 12318 --> 预测类别: 学生工作处
文档ID: 12319 --> 预测类别: 学生工作处
文档ID: 12320 --> 预测类别: 科技处
文档ID: 12277 --> 预测类别: 科技处
文档ID: 12278 --> 预测类别: 国际交流与合作处
文档ID: 12279 --> 预测类别: 学生工作处
文档ID: 12280 --> 预测类别: 国际交流与合作处
文档ID: 12281 --> 预测类别: 科技处
文档ID: 